In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

load_dotenv()

password = quote_plus(os.getenv("DB_PASSWORD"))
user = os.getenv("DB_USER", "root")
host = os.getenv("DB_HOST", "localhost")
db = os.getenv("DB_NAME", "olist_analytics")

engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")

with engine.connect() as conn:
    tables = ["customers", "sellers", "geolocation", "products", "orders",
              "order_items", "order_payments", "order_reviews", "order_item_totals"]
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        print(f"{table:20s} {result.scalar():,}")

customers            99,441
sellers              3,095
geolocation          19,015
products             32,951
orders               99,441
order_items          112,650
order_payments       103,886
order_reviews        98,408
order_item_totals    98,666


In [4]:
import pandas as pd

query = """
WITH first_purchase AS (
    SELECT
        c.customer_unique_id,
        DATE_FORMAT(MIN(o.order_purchase_timestamp), '%Y-%m') AS cohort_month
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.is_delivered = 1
    GROUP BY c.customer_unique_id
),
orders_with_cohort AS (
    SELECT
        fp.cohort_month,
        c.customer_unique_id,
        DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS order_month
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN first_purchase fp ON c.customer_unique_id = fp.customer_unique_id
    WHERE o.is_delivered = 1
),
cohort_activity AS (
    SELECT
        cohort_month,
        TIMESTAMPDIFF(MONTH, STR_TO_DATE(CONCAT(cohort_month, '-01'), '%Y-%m-%d'), STR_TO_DATE(CONCAT(order_month, '-01'), '%Y-%m-%d')) AS month_number,
        COUNT(DISTINCT customer_unique_id) AS active_customers
    FROM orders_with_cohort
    GROUP BY cohort_month, month_number
)
SELECT cohort_month, month_number, active_customers
FROM cohort_activity
ORDER BY cohort_month, month_number;
"""

df = pd.read_sql(query, engine)

cohort_sizes = df[df["month_number"] == 0].set_index("cohort_month")["active_customers"]
pivot = df.pivot(index="cohort_month", columns="month_number", values="active_customers")
retention = pivot.divide(cohort_sizes, axis=0).round(3) * 100

print(retention)

month_number     0      1    2    3    4    5    6    7    8    9    10   11  \
cohort_month                                                                   
2016-09       100.0    NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2016-10       100.0    NaN  NaN  NaN  NaN  NaN  0.4  NaN  NaN  0.4  NaN  0.4   
2016-12       100.0  100.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2017-01       100.0    0.3  0.3  0.1  0.4  0.1  0.4  0.1  0.1  NaN  0.4  0.1   
2017-02       100.0    0.2  0.3  0.1  0.4  0.1  0.2  0.2  0.1  0.2  0.1  0.3   
2017-03       100.0    0.4  0.4  0.4  0.4  0.2  0.2  0.3  0.3  0.1  0.4  0.1   
2017-04       100.0    0.6  0.2  0.2  0.3  0.3  0.4  0.3  0.3  0.2  0.3  0.1   
2017-05       100.0    0.5  0.5  0.3  0.3  0.3  0.4  0.1  0.3  0.3  0.3  0.3   
2017-06       100.0    0.5  0.4  0.4  0.3  0.4  0.4  0.2  0.1  0.2  0.3  0.4   
2017-07       100.0    0.5  0.3  0.2  0.3  0.2  0.3  0.1  0.2  0.3  0.2  0.3   
2017-08       100.0    0.7  0.3  0.3  0.